In [1]:
import io
import zipfile
import requests
import frontmatter

In [2]:
import requests
resp = requests.get('https://httpbin.org/get', timeout=10)
print(resp.status_code)

200


In [3]:
url = 'https://github.com/DataTalksClub/faq/archive/refs/heads/main.zip'
resp = requests.get(url, timeout=30)
print(resp.status_code)

200


In [4]:
def read_repo_data(repo_owner, repo_name):
    """
    Download and parse all markdown files from a GitHub repository.
    
    Args:
        repo_owner: GitHub username or organization
        repo_name: Repository name
    
    Returns:
        List of dictionaries containing file content and metadata
    """
    prefix = 'https://codeload.github.com' 
    url = f'{prefix}/{repo_owner}/{repo_name}/zip/refs/heads/main'
    resp = requests.get(url)
    
    if resp.status_code != 200:
        raise Exception(f"Failed to download repository: {resp.status_code}")

    repository_data = []
    zf = zipfile.ZipFile(io.BytesIO(resp.content))
    
    for file_info in zf.infolist():
        filename = file_info.filename
        filename_lower = filename.lower()

        if not (filename_lower.endswith('.md') 
            or filename_lower.endswith('.mdx')):
            continue
    
        try:
            with zf.open(file_info) as f_in:
                content = f_in.read().decode('utf-8', errors='ignore')
                post = frontmatter.loads(content)
                data = post.to_dict()
                data['filename'] = filename
                repository_data.append(data)
        except Exception as e:
            print(f"Error processing {filename}: {e}")
            continue
    
    zf.close()
    return repository_data


In [5]:
dtc_faq = read_repo_data('DataTalksClub', 'faq')
print(dtc_faq[0])

{'description': 'Review and process open FAQ PRs', 'content': 'Go through all open pull requests one by one. For each PR:\n\n## 1. Show Details\n- PR number and title\n- Course and section (from PR body)\n- Related issue number\n- **ALWAYS check sort_order**: List files in target section (`ls _questions/<course>/<section>/`) to find highest number, verify PR uses next sequential\n- Full diff (use `gh pr diff <number>`)\n\n## 2. Check Against These Rules\n\n### Section Placement\n- **Kestra questions** → must be in `module-2` (workflow orchestration), NOT `general` or `module-1`\n- **Docker + Kestra** → still `module-2` (Kestra is primary topic)\n- **Docker-only** (pgAdmin, Postgres, etc.) → `module-1`\n\n### Relevance (Close If)\n- Basic Linux/SQL tutorials (vim installation, SQL JOIN concepts, etc.)\n- Generic programming not tied to course content\n- Already exists in DE zoomcamp when proposed for ML zoomcamp\n\n### Duplicates (Check Before Creating)\n- Search existing FAQs: `grep -r

In [6]:
evidently_docs = read_repo_data('evidentlyai', 'docs')

In [7]:
print(f"Evidently docs: {len(evidently_docs)}")

Evidently docs: 95


In [8]:
doc = evidently_docs[45]
print(doc.keys())
print(f"Content length: {len(doc['content'])} characters")

dict_keys(['title', 'description', 'content', 'filename'])
Content length: 21712 characters


In [9]:
def sliding_window(seq, size, step):
    if size <= 0 or step <= 0:
        raise ValueError("size and step must be positive")

    n = len(seq)
    result = []
    for i in range(0, n, step):
        chunk = seq[i:i+size]
        result.append({'start': i, 'chunk': chunk})
        if i + size >= n:
            break

    return result

In [10]:
test = sliding_window(evidently_docs[45]['content'], 2000, 1000)
print(f"Number of chunks: {len(test)}")
print(f"First chunk start: {test[0]['start']}")
print(f"First chunk length: {len(test[0]['chunk'])}")

Number of chunks: 21
First chunk start: 0
First chunk length: 2000


In [11]:
import re

def split_markdown_by_level(text, level=2):
    """
    Split markdown text by a specific header level.
    
    :param text: Markdown text as a string
    :param level: Header level to split on
    :return: List of sections as strings
    """
    # This regex matches markdown headers
    # For level 2, it matches lines starting with "## "
    header_pattern = r'^(#{' + str(level) + r'} )(.+)$'
    pattern = re.compile(header_pattern, re.MULTILINE)

    # Split and keep the headers
    parts = pattern.split(text)
    
    sections = []
    for i in range(1, len(parts), 3):
        # We step by 3 because regex.split() with
        # capturing groups returns:
        # [before_match, group1, group2, after_match, ...]
        # here group1 is "## ", group2 is the header text
        header = parts[i] + parts[i+1]  # "## " + "Title"
        header = header.strip()

        # Get the content after this header
        content = ""
        if i+2 < len(parts):
            content = parts[i+2].strip()

        if content:
            section = f'{header}\n\n{content}'
        else:
            section = header
        sections.append(section)
    
    return sections


In [12]:
# text = evidently_docs[45]['content']
# sections = split_markdown_by_level(text, level=2)
# print(f"Number of sections: {len(sections)}")
# print(sections[0])

Number of sections: 8
## 1. Installation and Imports

Install Evidently:

```python
pip install evidently[llm] 
```

Import the required modules:

```python
import pandas as pd
from evidently.future.datasets import Dataset
from evidently.future.datasets import DataDefinition
from evidently.future.datasets import Descriptor
from evidently.future.descriptors import *
from evidently.future.report import Report
from evidently.future.presets import TextEvals
from evidently.future.metrics import *
from evidently.future.tests import *

from evidently.features.llm_judge import BinaryClassificationPromptTemplate
```

To connect to Evidently Cloud:

```python
from evidently.ui.workspace.cloud import CloudWorkspace
```

**Optional.** To create monitoring panels as code:

```python
from evidently.ui.dashboards import DashboardPanelPlot
from evidently.ui.dashboards import DashboardPanelTestSuite
from evidently.ui.dashboards import DashboardPanelTestSuiteCounter
from evidently.ui.dashboards import T

In [13]:
# evidently_chunks = []

# for doc in evidently_docs:
#     doc_copy = doc.copy()
#     doc_content = doc_copy.pop('content')
#     sections = split_markdown_by_level(doc_content, level=2)
#     for section in sections:
#         section_doc = doc_copy.copy()
#         section_doc['section'] = section
#         evidently_chunks.append(section_doc)

In [14]:
# print(f"Total chunks: {len(evidently_chunks)}")
# print(evidently_chunks[0].keys())

Total chunks: 266
dict_keys(['title', 'description', 'filename', 'section'])


In [15]:
# from groq import Groq
# from dotenv import load_dotenv
# import os

# load_dotenv()

# groq_client = Groq(api_key=os.getenv("GROQ_API_KEY"))

# def llm(prompt, model='llama-3.3-70b-versatile'):
#     messages = [
#         {"role": "user", "content": prompt}
#     ]
    
#     response = groq_client.chat.completions.create(
#         model=model,
#         messages=messages
#     )
    
#     return response.choices[0].message.content

In [16]:
# print(llm("Say hello in one sentence."))

Hello, it's nice to meet you and I'm here to assist you with any questions or topics you'd like to discuss.


In [17]:
prompt_template = """
Split the provided document into logical sections
that make sense for a Q&A system.

Each section should be self-contained and cover
a specific topic or concept.

IMPORTANT: You MUST separate each section with --- on its own line. Do not skip this.

<DOCUMENT>
{document}
</DOCUMENT>

Use this format:

## Section Name

Section content with all relevant details

---

## Another Section Name

Another section content

---
""".strip()


In [18]:
def intelligent_chunking(text):
    prompt = prompt_template.format(document=text)
    response = llm(prompt)
    sections = response.split('---')
    sections = [s.strip() for s in sections if s.strip()]
    return sections

In [19]:
# test_sections = intelligent_chunking(evidently_docs[45]['content'])
# print(f"Number of sections: {len(test_sections)}")
# print(test_sections[0])

Number of sections: 11
## Introduction to Regression Testing for LLM Outputs

In this tutorial, you will learn how to perform regression testing for LLM outputs. You can compare new and old responses after changing a prompt, model, or anything else in your system. By re-running the same inputs with new parameters, you can spot any significant changes. This helps you push updates with confidence or identify issues to fix.


In [20]:
# from tqdm.auto import tqdm

# evidently_chunks_llm = []

# for doc in tqdm(evidently_docs[:5]):  # just first 5 docs
#     doc_copy = doc.copy()
#     doc_content = doc_copy.pop('content')
#     sections = intelligent_chunking(doc_content)
#     for section in sections:
#         section_doc = doc_copy.copy()
#         section_doc['section'] = section
#         evidently_chunks_llm.append(section_doc)

# print(f"Total chunks: {len(evidently_chunks_llm)}")

  0%|          | 0/5 [00:00<?, ?it/s]

Total chunks: 26


In [21]:
from minsearch import Index

print(len(evidently_chunks))
print(evidently_chunks[0].keys())

266
dict_keys(['title', 'description', 'filename', 'section'])


In [22]:
evidently_docs = read_repo_data('evidentlyai', 'docs')

evidently_chunks = []

for doc in evidently_docs:
    doc_copy = doc.copy()
    doc_content = doc_copy.pop('content')
    chunks = sliding_window(doc_content, 2000, 1000)
    for chunk in chunks:
        chunk.update(doc_copy)
    evidently_chunks.extend(chunks)

In [23]:
print(len(evidently_chunks))
print(evidently_chunks[0].keys())

576
dict_keys(['start', 'chunk', 'title', 'description', 'filename'])


In [24]:
from minsearch import Index

index = Index(
    text_fields=["chunk", "title", "description", "filename"],
    keyword_fields=[]
)

index.fit(evidently_chunks)

In [25]:
query = 'What should be in a test dataset for AI evaluation?'
results = index.search(query)
print(f"Number of results: {len(results)}")
print(results[0])

Number of results: 10
{'start': 0, 'chunk': 'Retrieval-Augmented Generation (RAG) systems rely on retrieving answers from a knowledge base before generating responses. To evaluate them effectively, you need a test dataset that reflects what the system *should* know.\n\nInstead of manually creating test cases, you can generate them directly from your knowledge source, ensuring accurate and relevant ground truth data.\n\n## Create a RAG test dataset\n\nYou can generate ground truth RAG dataset from your data source.\n\n### 1. Create a Project\n\nIn the Evidently UI, start a new Project or open an existing one.\n\n* Navigate to “Datasets” in the left menu.\n* Click “Generate” and select the “RAG” option.\n\n![](/images/synthetic/synthetic_data_select_method.png)\n\n### 2. Upload your knowledge base\n\nSelect a file containing the information your AI system retrieves from. Supported formats: Markdown (.md), CSV, TXT, PDFs. Choose how many inputs to generate.\n\n![](/images/synthetic/synthe

In [26]:
dtc_faq = read_repo_data('DataTalksClub', 'faq')
de_dtc_faq = [d for d in dtc_faq if 'data-engineering' in d['filename']]

In [27]:
de_dtc_faq

[{'id': '9e508f2212',
  'question': 'Course: When does the course start?',
  'sort_order': 1,
  'content': "The next cohort starts January 12th, 2026. More info at [DTC](https://datatalks.club/blog/guide-to-free-online-courses-at-datatalks-club.html).\n\n- Register before the course starts using this [link](https://airtable.com/shr6oVXeQvSI5HuWD).\n- Join the [course Telegram channel with announcements](https://t.me/dezoomcamp).\n- Don’t forget to register in DataTalks.Club's Slack and [join](https://datatalks.club/docs/general/slack/) the channel.",
  'filename': 'faq-main/_questions/data-engineering-zoomcamp/general/001_9e508f2212_course-when-does-the-course-start.md'},
 {'id': 'bfafa427b3',
  'question': 'Course: What are the prerequisites for this course?',
  'sort_order': 2,
  'content': 'To get the most out of this course, you should have:\n\n- Basic coding experience\n- Familiarity with SQL\n- Experience with Python (helpful but not required)\n\nNo prior data engineering experie

In [28]:
faq_index = Index(
    text_fields=['question','content'],  # what fields does FAQ have?
    keyword_fields=[]
)
faq_index.fit(de_dtc_faq)

In [29]:
results = faq_index.search('Can I join the course now?')
print(f"Results: {len(results)}")
print(results[0])

Results: 10
{'id': '3f1424af17', 'question': 'Course: Can I still join the course after the start date?', 'sort_order': 3, 'content': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.", 'filename': 'faq-main/_questions/data-engineering-zoomcamp/general/003_3f1424af17_course-can-i-still-join-the-course-after-the-start.md'}


In [30]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer('multi-qa-distilbert-cos-v1')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/523 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/333 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [31]:
record = de_dtc_faq[2]
text = record['question'] + ' ' + record['content']
v_doc = embedding_model.encode(text)

query = 'I just found out about the course. Can I enroll now?'
v_query = embedding_model.encode(query)

similarity = v_query.dot(v_doc)
print(f"Vector size: {v_doc.shape}")
print(f"Similarity: {similarity}")

Vector size: (768,)
Similarity: 0.5190930962562561


In [33]:
print(record)

{'id': '3f1424af17', 'question': 'Course: Can I still join the course after the start date?', 'sort_order': 3, 'content': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.", 'filename': 'faq-main/_questions/data-engineering-zoomcamp/general/003_3f1424af17_course-can-i-still-join-the-course-after-the-start.md'}


In [34]:
from minsearch import VectorSearch
from tqdm.auto import tqdm
import numpy as np

faq_embeddings = []

for record in de_dtc_faq:
    text = record['question'] + ' ' + record['content']
    v_doc = embedding_model.encode(text)
    faq_embeddings.append(v_doc)

faq_embeddings = np.array(faq_embeddings)

faq_vindex = VectorSearch()
faq_vindex.fit(faq_embeddings, de_dtc_faq)

In [35]:
query = 'Can I join the course now?'
q = embedding_model.encode(query)
results = faq_vindex.search(q)
print(f"Results: {len(results)}")
print(results[0])

Results: 10
{'id': '3f1424af17', 'question': 'Course: Can I still join the course after the start date?', 'sort_order': 3, 'content': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.", 'filename': 'faq-main/_questions/data-engineering-zoomcamp/general/003_3f1424af17_course-can-i-still-join-the-course-after-the-start.md'}


In [36]:
query = 'I just discovered the program, can I still enroll?'

# text search
text_results = faq_index.search(query)
print("TEXT SEARCH:")
print(text_results[0]['question'])

# vector search
q = embedding_model.encode(query)
vector_results = faq_vindex.search(q)
print("\nVECTOR SEARCH:")
print(vector_results[0]['question'])

TEXT SEARCH:
Course: Can I still join the course after the start date?

VECTOR SEARCH:
Course: Can I still join the course after the start date?


In [37]:
def text_search(query):
    return faq_index.search(query, num_results=5)

def vector_search(query):
    q = embedding_model.encode(query)
    return faq_vindex.search(q, num_results=5)

def hybrid_search(query):
    text_results = text_search(query)
    vector_results = vector_search(query)
    
    # Combine and deduplicate results
    seen_ids = set()
    combined_results = []

    for result in text_results + vector_results:
        if result['filename'] not in seen_ids:
            seen_ids.add(result['filename'])
            combined_results.append(result)
    
    return combined_results


In [38]:
query = 'Can I join the course now?'

print(f"Text search: {len(text_search(query))} results")
print(f"Vector search: {len(vector_search(query))} results")
print(f"Hybrid search: {len(hybrid_search(query))} results")

Text search: 5 results
Vector search: 5 results
Hybrid search: 6 results
